### 1. LIBRERÍAS (Cargue e Instalación)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display # Importamos la función display
import seaborn as sns # Importamos seaborn para las visualizaciones de correlación

# Install fpdf library if not already installed
try:
    from fpdf import FPDF
except ImportError:
    !pip install fpdf
    from fpdf import FPDF

import sqlite3
import matplotlib.ticker as mticker

# Importar la función files de google.colab para la descarga
from google.colab import files

### 2. CARGUE DE ARCHIVO CSV

In [ ]:
def procesar_dataset_simulado(archivo_csv):
    """Carga y limpia el dataset, transformando el formato regional a numérico.

    Args:
        archivo_csv (str): La ruta al archivo CSV simulado.

    Returns:
        pd.DataFrame: El DataFrame procesado con los datos limpios.

    Raises:
        FileNotFoundError: Si no se encontró el archivo CSV.
    """
    if not os.path.exists(archivo_csv):
        raise FileNotFoundError(f"No se encontró el archivo simulado: {archivo_csv}. Ejecute el script generador primero.")

    df = pd.read_csv(archivo_csv, sep=';')

    # Limpieza de formato numérico (de '1234,56' a 1234.56) para columnas de moneda
    cols_moneda = ["BAC", "PV", "EV", "AC"]
    for col in cols_moneda:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].str.replace(',', '.').astype(float)
        elif col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Debugging prints for PV, EV, AC at SEM_CORTE after loading
    sem_corte_local = globals().get('SEM_CORTE', 20)
    df_filtered_at_corte = df[df['Semana'] == sem_corte_local]

    if not df_filtered_at_corte.empty:
        pv_sum_loaded = df_filtered_at_corte['PV'].sum()
        ev_sum_loaded = df_filtered_at_corte['EV'].sum()
        ac_sum_loaded = df_filtered_at_corte['AC'].sum()
        print(f"[DEBUG - Carga de Datos] Sumas en Semana {sem_corte_local}: PV={pv_sum_loaded:,.0f}, EV={ev_sum_loaded:,.0f}, AC={ac_sum_loaded:,.0f})")
    else:
        print(f"[DEBUG - Carga de Datos] No hay datos para la Semana {sem_corte_local} en df_loaded.")

    return df

### 3. CREACION DE TRABLAS Y BASE DE DATOS

In [ ]:
print("--- Métricas de Desempeño Global del Proyecto ---")
print(f"BAC (Budget At Completion): ${globals().get('total_project_bac_mc', 0.0):,.0f}")
print(f"CPI Global (Cost Performance Index): {globals().get('global_cpi_for_pdf', 0.0):.2f}")
print(f"SPI Global (Schedule Performance Index): {globals().get('global_spi_for_pdf', 0.0):.2f}")
print(f"EAC (Estimación a la Conclusión): ${globals().get('eac_cpi_global', 0.0):,.0f}")
print(f"ETC (Estimación para Terminar): ${globals().get('etc_cpi_global', 0.0):,.0f}")
print(f"VAC (Variación a la Conclusión): ${globals().get('vac_cpi_global', 0.0):,.0f}")
tcpi_bac_val = globals().get('tcpi_bac_global', np.nan)
tcpi_eac_val = globals().get('tcpi_eac_global', np.nan)
print(f"TCPI (basado en BAC): {tcpi_bac_val:.2f}" if not np.isnan(tcpi_bac_val) else "TCPI (basado en BAC): N/A")
print(f"TCPI (basado en EAC): {tcpi_eac_val:.2f}" if not np.isnan(tcpi_eac_val) else "TCPI (basado en EAC): N/A")

In [ ]:
def create_and_populate_sqlite_db(df_source, db_file):
    """Crea el archivo de la base de datos SQLite, define los esquemas de las tablas
    y carga los datos desde df_source en las tablas correspondientes.
    """
    # Eliminar el archivo de la base de datos si ya existe para empezar de cero
    if os.path.exists(db_file):
        os.remove(db_file)
        print(f"Base de datos existente '{db_file}' eliminada.")

    conn = None
    try:
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        print(f"Base de datos '{db_file}' creada exitosamente.")

        # 2. Crear la tabla Dim_WBS
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Dim_WBS (
                WBS TEXT PRIMARY KEY,
                Capitulo TEXT,
                Subcapitulo TEXT,
                Actividad TEXT
            )
        """
        )
        print("Tabla 'Dim_WBS' creada.")

        # 3. Crear la tabla Fact_Seguimiento
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Fact_Seguimiento (
                WBS TEXT,
                Semana INTEGER,
                EV INTEGER,
                AC INTEGER,
                Comentario_Operativo TEXT,
                Comentario_Administrativo TEXT,
                FOREIGN KEY (WBS) REFERENCES Dim_WBS(WBS)
            )
        """
        )
        print("Tabla 'Fact_Seguimiento' creada.")

        # 4. Crear la tabla Fact_Proyecto
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Fact_Proyecto (
                WBS TEXT,
                Semana INTEGER,
                PV INTEGER,
                BAC INTEGER,
                FOREIGN KEY (WBS) REFERENCES Dim_WBS(WBS)
            )
        """
        )
        print("Tabla 'Fact_Proyecto' creada.")

        # Extraer datos y cargarlos
        df_dim_wbs = df_source[['WBS', 'Capitulo', 'Subcapitulo', 'Actividad']].drop_duplicates().copy()
        df_fact_seguimiento = df_source[['WBS', 'Semana', 'EV', 'AC', 'Comentario_Operativo', 'Comentario_Administrativo']].copy()
        df_fact_proyecto = df_source[['WBS', 'Semana', 'PV', 'BAC']].copy()

        df_dim_wbs.to_sql('Dim_WBS', conn, if_exists='append', index=False)
        df_fact_seguimiento.to_sql('Fact_Seguimiento', conn, if_exists='append', index=False)
        df_fact_proyecto.to_sql('Fact_Proyecto', conn, if_exists='append', index=False)

        conn.commit()
        print("Datos cargados en las tablas SQLite.")

    except sqlite3.Error as e:
        print(f"Error al crear o poblar la base de datos: {e}")
    finally:
        if conn:
            conn.close()

def load_dimensional_model_from_sqlite(db_file):
    """Carga el modelo dimensional completo desde la base de datos SQLite.

    Returns:
        pd.DataFrame: El DataFrame combinado con todos los datos.
    """
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        df_dim_wbs = pd.read_sql_query("SELECT * FROM Dim_WBS", conn)
        df_fact_seguimiento = pd.read_sql_query("SELECT * FROM Fact_Seguimiento", conn)
        df_fact_proyecto = pd.read_sql_query("SELECT * FROM Fact_Proyecto", conn)

        df_merged_facts = pd.merge(df_fact_seguimiento, df_fact_proyecto,
                                 on=['WBS', 'Semana'],
                                 how='inner')
        df_modelo_dimensional = pd.merge(df_merged_facts, df_dim_wbs,
                                       on='WBS',
                                       how='inner')
        return df_modelo_dimensional
    except sqlite3.Error as e:
        print(f"Error al cargar el modelo dimensional desde SQLite: {e}")
        return pd.DataFrame()
    finally:
        if conn:
            conn.close()

### 4. CÁLCULOS

In [ ]:
def calcular_indicadores_por_capitulo(df, sem_corte):
    """Calcula el CPI y SPI agregado por frente de obra constructivo.

    Args:
        df (pd.DataFrame): El DataFrame de proyecto con los datos cargados.
        sem_corte (int): La semana de corte para el análisis.

    Returns:
        pd.DataFrame: Un DataFrame con los KPIs de valor ganado calculados por capítulo.
    """
    # Calculate BAC per chapter from the full dataset, considering unique WBS BACs
    df_unique_wbs_bac = df[['WBS', 'Capitulo', 'BAC']].drop_duplicates(subset=['WBS'])
    bac_por_capitulo = df_unique_wbs_bac.groupby('Capitulo')['BAC'].sum().reset_index()
    bac_por_capitulo = bac_por_capitulo.rename(columns={'BAC': 'BAC_Capitulo_Total'})

    df_filtered_at_corte_for_kpis = df[df['Semana'] == sem_corte].copy()
    resumen = df_filtered_at_corte_for_kpis.groupby('Capitulo')[["PV", "EV", "AC"]].sum().reset_index()

    # Merge the calculated BAC_Capitulo_Total into the resumen DataFrame
    resumen = pd.merge(resumen, bac_por_capitulo, on='Capitulo', how='left')
    resumen = resumen.rename(columns={'BAC_Capitulo_Total': 'BAC'})

    resumen['CPI'] = np.where(resumen['AC'] == 0, 0, resumen['EV'] / resumen['AC'])
    resumen['SPI'] = np.where(resumen['PV'] == 0, 0, resumen['EV'] / resumen['PV'])
    return resumen

In [ ]:
def calcular_kpis_globales(df_modelo_dimensional, sem_corte):
    """Calcula los KPIs globales del proyecto y los guarda en el scope global."""

    df_at_corte_global = df_modelo_dimensional[df_modelo_dimensional['Semana'] == sem_corte].copy()

    total_pv_global = df_at_corte_global['PV'].sum()
    total_ev_global = df_at_corte_global['EV'].sum()
    total_ac_global = df_at_corte_global['AC'].sum()

    # Calculate total_project_bac_mc (BAC from unique WBS entries)
    df_unique_wbs_bac = df_modelo_dimensional[['WBS', 'BAC']].drop_duplicates(subset=['WBS'])
    total_project_bac_mc = df_unique_wbs_bac['BAC'].sum()

    # CPI Global
    global_cpi_for_pdf = total_ev_global / total_ac_global if total_ac_global != 0 else 0.0

    # SPI Global
    global_spi_for_pdf = total_ev_global / total_pv_global if total_pv_global != 0 else 0.0

    # EAC (Estimate At Completion)
    # Using CPI
    eac_cpi_global = total_ac_global + (total_project_bac_mc - total_ev_global) / global_cpi_for_pdf if global_cpi_for_pdf != 0 else total_project_bac_mc

    # ETC (Estimate To Complete)
    etc_cpi_global = eac_cpi_global - total_ac_global

    # VAC (Variance At Completion)
    vac_cpi_global = total_project_bac_mc - eac_cpi_global

    # TCPI (To Complete Performance Index)
    tcpi_bac_denom = (total_project_bac_mc - total_ac_global)
    tcpi_eac_denom = (eac_cpi_global - total_ac_global)

    print(f"[DEBUG - TCPI] total_project_bac_mc: {total_project_bac_mc:,.0f}")
    print(f"[DEBUG - TCPI] total_ev_global: {total_ev_global:,.0f}")
    print(f"[DEBUG - TCPI] total_ac_global: {total_ac_global:,.0f}")
    print(f"[DEBUG - TCPI] eac_cpi_global: {eac_cpi_global:,.0f}")
    print(f"[DEBUG - TCPI] Denominator for TCPI (BAC): {tcpi_bac_denom:,.0f}")
    print(f"[DEBUG - TCPI] Denominator for TCPI (EAC): {tcpi_eac_denom:,.0f}")

    tcpi_bac_global = (total_project_bac_mc - total_ev_global) / tcpi_bac_denom if tcpi_bac_denom != 0 else np.nan
    tcpi_eac_global = (total_project_bac_mc - total_ev_global) / tcpi_eac_denom if tcpi_eac_denom != 0 else np.nan

    # Store in globals for access by other functions/cells
    globals()['total_project_bac_mc'] = total_project_bac_mc
    globals()['global_cpi_for_pdf'] = global_cpi_for_pdf
    globals()['global_spi_for_pdf'] = global_spi_for_pdf
    globals()['eac_cpi_global'] = eac_cpi_global
    globals()['etc_cpi_global'] = etc_cpi_global
    globals()['vac_cpi_global'] = vac_cpi_global
    globals()['tcpi_bac_global'] = tcpi_bac_global
    globals()['tcpi_eac_global'] = tcpi_eac_global

    print(f"[DEBUG - KPIs Globales] BAC: {total_project_bac_mc:,.0f}, PV: {total_pv_global:,.0f}, EV: {total_ev_global:,.0f}, AC: {total_ac_global:,.0f}")
    print(f"[DEBUG - KPIs Globales] CPI: {global_cpi_for_pdf:.2f}, SPI: {global_spi_for_pdf:.2f}, EAC: {eac_cpi_global:,.0f}")
    print(f"[DEBUG - KPIs Globales] ETC: {etc_cpi_global:,.0f}, VAC: {vac_cpi_global:,.0f}, TCPI (BAC): {tcpi_bac_global:.2f} (EAC): {tcpi_eac_global:.2f}")

### 5. GRÁFICOS

In [ ]:
def graficar_curva_s_dimensional(df_input, semana_corte, filename='s_curve_plot.png'):
    """Renderiza la Curva S leyendo directamente la proyección de la Fact Table.
    Utiliza los datos de PV, EV, AC ya presentes y agrega por semana.
    """
    df_plot = df_input.groupby('Semana').agg({'PV': 'sum', 'EV': 'sum', 'AC': 'sum'}).reset_index()

    # Ocultar futuros para no distorsionar la gráfica
    df_plot.loc[df_plot['Semana'] > semana_corte, ['EV', 'AC']] = np.nan

    plt.figure(figsize=(8, 6))
    plt.plot(df_plot['Semana'], df_plot['PV'], label='Valor Planificado (PV)', color='#1f77b4', linestyle='--', linewidth=2)
    plt.plot(df_plot['Semana'], df_plot['EV'], label='Valor Ganado (EV)', color='#2ca02c', linewidth=2.5)
    plt.plot(df_plot['Semana'], df_plot['AC'], label='Costo Real (AC)', color='#d62728', linewidth=2.5)

    plt.axvline(x=semana_corte, color='grey', linestyle=':', label=f'Corte S{semana_corte}')

    # Set x-axis ticks to integers
    plt.gca().xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # Anotaciones
    pv_values = df_plot.loc[df_plot['Semana'] == semana_corte, 'PV'].values
    ev_values = df_plot.loc[df_plot['Semana'] == semana_corte, 'EV'].values
    ac_values = df_plot.loc[df_plot['Semana'] == semana_corte, 'AC'].values

    pv_at_cut = pv_values[0] if pv_values.size > 0 else 0
    ev_at_cut = ev_values[0] if ev_values.size > 0 else 0
    ac_at_cut = ac_values[0] if ac_values.size > 0 else 0
    max_y = df_plot['PV'].max() if not df_plot['PV'].empty else 1

    # Adjusted PV label position
    plt.text(semana_corte + 0.5, pv_at_cut - 0.005*max_y, f'PV: ${pv_at_cut:,.0f}', color='#1f77b4', fontsize=11)
    plt.text(semana_corte + 0.5, ev_at_cut + 0.005*max_y, f'EV: ${ev_at_cut:,.0f}', color='#2ca02c', fontsize=11)
    plt.text(semana_corte + 0.5, ac_at_cut - 0.02*max_y, f'AC: ${ac_at_cut:,.0f}', color='#d62728', fontsize=11)

    # Zonas de brecha (Relleno)
    plt.fill_between(df_plot['Semana'], df_plot['EV'], df_plot['PV'], where=(df_plot['Semana'] <= semana_corte) & (df_plot['EV'] < df_plot['PV']), color='orange', alpha=0.1, label='Retraso')
    plt.fill_between(df_plot['Semana'], df_plot['EV'], df_plot['AC'], where=(df_plot['Semana'] <= semana_corte) & (df_plot['EV'] < df_plot['AC']), color='red', alpha=0.1, label='Sobrecosto')


    plt.title('Curva S de Avance de Proyecto', fontsize=16, fontweight='bold')
    plt.xlabel('Semana del Proyecto', fontsize=12)
    plt.ylabel('Inversión Acumulada ($)', fontsize=12)
    plt.legend(loc='upper left')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

def graficar_matriz_desempeno(chapter_kpis_df, filename='temp_matriz_cpispi.png'):
    """Genera y guarda el gráfico de matriz de desempeño CPI vs SPI.
    """
    cpi_vals = chapter_kpis_df['CPI'].tolist()
    spi_vals = chapter_kpis_df['SPI'].tolist()
    categorias = chapter_kpis_df['Capitulo'].tolist()

    plt.figure(figsize=(8, 6))
    plt.scatter(spi_vals, cpi_vals, color='navy', s=150, zorder=5)
    plt.axhline(1.0, color='red', linestyle='--', alpha=0.5)
    plt.axvline(1.0, color='red', linestyle='--', alpha=0.5)

    plt.fill_between([0, 1], 0, 1, color='red', alpha=0.1)
    plt.fill_between([1, 1.5], 1, 1.5, color='green', alpha=0.1)

    plt.xlim(0.5, 1.2)
    plt.ylim(0.5, 1.2)
    plt.xlabel('SPI (Eficiencia Cronograma)', fontsize=12)
    plt.ylabel('CPI (Eficiencia Costo)', fontsize=12)
    plt.title('Matriz de Desempeño (CPI vs SPI)', fontsize=16, fontweight='bold')

    for i, txt in enumerate(categorias):
        plt.annotate(txt, (spi_vals[i], cpi_vals[i]), xytext=(5,5), textcoords='offset points', fontsize=10)

    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

### 6. SIMULACIÓN DE MONTECARLO

In [ ]:
def simular_duraciones_frentes(iteraciones=10000):
    # Definimos la incertidumbre por frente (Semanas: Min, Moda, Max)
    # Basado en el impacto de Nivel Freático y Aduanas
    obras_civiles = np.random.triangular(28, 32, 48, iteraciones)
    suministros = np.random.triangular(35, 40, 55, iteraciones)
    montaje_pruebas = np.random.triangular(12, 15, 25, iteraciones)
    return obras_civiles, suministros, montaje_pruebas

def generar_reporte_montecarlo_completo(db_file, num_sims=10000):
    conn = sqlite3.connect(db_file)

    # Retrieve global variables or set defaults
    SEM_CORTE = globals().get('SEM_CORTE', 20)
    PLAZO = globals().get('PLAZO', 52)

    # 1. Extracción de Estado Financiero (para cálculo de costo)
    df_bac_project_total = pd.read_sql_query("SELECT WBS, BAC FROM Fact_Proyecto", conn).drop_duplicates(subset=['WBS'])
    total_project_bac = df_bac_project_total['BAC'].sum()

    df_seg_at_corte = pd.read_sql_query(f"SELECT WBS, EV, AC FROM Fact_Seguimiento WHERE Semana = {SEM_CORTE}", conn)
    total_ev = df_seg_at_corte['EV'].sum()
    total_ac = df_seg_at_corte['AC'].sum()

    # 2. SIMULACIÓN DE TIEMPO
    duraciones_obras_civiles, duraciones_suministros, duraciones_montaje_pruebas = simular_duraciones_frentes(num_sims)
    duraciones = np.maximum(duraciones_suministros, duraciones_obras_civiles + duraciones_montaje_pruebas)

    # 3. SIMULACIÓN DE COSTO
    total_cpi = total_ev / total_ac if total_ac > 0 else 1.0
    etc_base = (total_project_bac - total_ev) / total_cpi if total_cpi != 0 else (total_project_bac - total_ev) # Avoid division by zero

    # Costos (ETC USD) - Varianza alta por riesgos geológicos, usando distribución triangular
    c_o = etc_base * 0.90 # Optimista
    c_m = etc_base        # Más probable
    c_p = etc_base * 1.50 # Pesimista (mayor incertidumbre)

    # Asegurar que c_o < c_m < c_p
    c_o, c_m, c_p = sorted([c_o, c_m, c_p])
    if c_o == c_m: c_m = c_o * 1.05 # Ensure mode is distinct if o==m
    if c_m == c_p: c_p = c_m * 1.05 # Ensure pessimistic is distinct if m==p

    etc_sim = np.random.triangular(c_o, c_m, c_p, num_sims)
    costos_sim = total_ac + etc_sim

    # 4. Cálculo de Percentiles
    p10_t, p50_t, p90_t = np.percentile(duraciones, [10, 50, 90])
    p10_c, p50_c, p90_c = np.percentile(costos_sim, [10, 50, 90])

    # Global variables for PDF report
    globals()['p10_tiempo'] = p10_t
    globals()['p50_tiempo'] = p50_t
    globals()['p90_tiempo'] = p90_t
    globals()['p10_costo'] = p10_c
    globals()['p50_costo'] = p50_c
    globals()['p90_costo'] = p90_c
    globals()['total_project_bac_mc'] = total_project_bac

    # 5. GENERACIÓN DE GRÁFICOS LADO A LADO
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    # --- GRÁFICO 1: FRECUENCIA DE TIEMPO (VERDE #28A745) ---
    ax1.hist(duraciones, bins=40, color='#28A745', alpha=0.9, edgecolor='white')
    ax1.axvline(PLAZO, color='red', linestyle='--', linewidth=2, label=f'Meta: {PLAZO} sem')
    ax1.axvline(p10_t, color='black', linestyle=':', label=f'P10 (Optimista): {p10_t:.1f}')
    ax1.axvline(p50_t, color='blue', linestyle='-.', label=f'P50 (Mediana): {p50_t:.1f}')
    ax1.axvline(p90_t, color='darkred', linestyle='-', linewidth=2, label=f'P90 (Conservador): {p90_t:.1f}')
    ax1.set_title('ANÁLISIS SENSITIVO: FRECUENCIA DE TIEMPO', fontweight='bold', fontsize=14)
    ax1.set_xlabel('Semanas de Duración Final')
    ax1.set_ylabel('Frecuencia de Escenarios')
    ax1.legend()

    # --- GRÁFICO 2: FRECUENCIA DE COSTO (NARANJA #F39C3E) ---
    ax2.hist(costos_sim, bins=40, color='#F39C3E', alpha=0.9, edgecolor='white')
    ax2.axvline(total_project_bac, color='red', linestyle='--', linewidth=2, label=f'Presupuesto (BAC): ${total_project_bac:,.0f}')
    ax2.axvline(p10_c, color='black', linestyle=':', label=f'P10 (Optimista): ${p10_c:,.0f}')
    ax2.axvline(p50_c, color='blue', linestyle='-.', label=f'P50 (Mediana): ${p50_c:,.0f}')
    ax2.axvline(p90_c, color='darkred', linestyle='-', linewidth=2, label=f'P90 (Conservador): ${p90_c:,.0f}')
    ax2.set_title('ANÁLISIS SENSITIVO: FRECUENCIA DE COSTO', fontweight='bold', fontsize=14)
    ax2.set_xlabel('Costo Final Proyectado (USD)')
    ax2.set_ylabel('Frecuencia de Escenarios')
    ax2.ticklabel_format(style='plain', axis='x') # Prevent scientific notation on x-axis
    ax2.legend()

    plt.tight_layout()
    plt.savefig('montecarlo_tiempo_costo.png', dpi=300)
    plt.close()

    # Probabilidades de éxito por ruta crítica
    prob_success_path1 = (duraciones_suministros <= PLAZO).mean() * 100
    globals()['prob_success_path1'] = prob_success_path1 # Make it available globally for PDF
    prob_success_path2 = (duraciones_obras_civiles + duraciones_montaje_pruebas <= PLAZO).mean() * 100
    globals()['prob_success_path2'] = prob_success_path2 # Make it available globally for PDF

    rutas = ['Ruta de Suministros', 'Ruta de Obras Civiles + Montaje + Pruebas']
    probabilidades = [prob_success_path1, prob_success_path2]

    plt.figure(figsize=(10, 6))
    bars = plt.bar(rutas, probabilidades, color=['#3498db', '#2ecc71'])
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f'{yval:.2f}%', ha='center', va='bottom')
    plt.title('Probabilidad de Éxito por Ruta Crítica (Meta: 52 Semanas)', fontsize=14, fontweight='bold')
    plt.xlabel('Ruta Crítica')
    plt.ylabel('Probabilidad de Éxito (%)')
    plt.ylim(0, 105) # Increased y-axis limit to prevent label overlap
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('probabilidad_exito_rutas.png', dpi=300)
    plt.close()
    print("Gráfico de barras de probabilidades de éxito generado: 'probabilidad_exito_rutas.png'")

    # ACTUALIZACIÓN DE BASE DE DATOS (TABLAS mc_)
    df_percentiles = pd.DataFrame([ # Updated to include cost
        {'metrica': 'Meta_Contractual', 'semanas': PLAZO, 'Costo_USD': total_project_bac},
        {'metrica': 'P10_Optimista_Tiempo', 'semanas': p10_t, 'Costo_USD': p10_c},
        {'metrica': 'P50_Mediana_Tiempo', 'semanas': p50_t, 'Costo_USD': p50_c},
        {'metrica': 'P90_Conservador_Tiempo', 'semanas': p90_t, 'Costo_USD': p90_c}
    ])
    df_percentiles.to_sql('mc_percentiles', conn, if_exists='replace', index=False)

    # Frecuencias de Tiempo
    frec_t, bordes_t = np.histogram(duraciones, bins=range(int(duraciones.min())-1, int(duraciones.max())+2))
    pd.DataFrame({'semana': bordes_t[:-1], 'frecuencia': frec_t}).to_sql('mc_frecuencias_tiempo', conn, if_exists='replace', index=False)

    # Frecuencias de Costo
    min_cost_bin = int(costos_sim.min() / 1e6) * 1e6 # Round down to nearest million
    max_cost_bin = (int(costos_sim.max() / 1e6) + 1) * 1e6 # Round up to nearest million
    cost_bins = np.linspace(min_cost_bin, max_cost_bin, 20)
    frec_c, bordes_c = np.histogram(costos_sim, bins=cost_bins)
    pd.DataFrame({'costo': bordes_c[:-1], 'frecuencia': frec_c}).to_sql('mc_frecuencias_costo', conn, if_exists='replace', index=False)

    # Probabilidades de Ruta
    df_rutas = pd.DataFrame({
        'ruta': ['Ruta de Suministros', 'Ruta de Obras Civiles + Montaje + Pruebas'],
        'probabilidad_exito': [prob_success_path1, prob_success_path2]
    })
    df_rutas.to_sql('mc_probabilidades_ruta', conn, if_exists='replace', index=False)

    conn.close()
    print("Gráficos y tablas de Monte Carlo (tiempo y costo) actualizados con éxito.")

### 6. GENERACIÓN DE INFORME

In [ ]:
!pip install fpdf
import re
from fpdf import FPDF # Added this import statement

class InformePMO(FPDF):
    """Clase personalizada para generar informes PDF con encabezados y pies de página específicos.
    """
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.set_text_color(0, 51, 102)
        self.cell(0, 10, 'PROJECT MANAGEMENT OFFICE (PMO)', border=0, ln=1, align='C')
        self.set_font('Arial', 'I', 11)
        self.set_text_color(128, 128, 128)
        self.cell(0, 5, 'Informe Ejecutivo de Estado de Proyecto - Arquitectura Dimensional', border='B', ln=1, align='C')
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(128, 128, 128)
        self.set_x(10)
        self.cell(0, 10, 'Fuente: project_data.db', 0, 0, 'L')
        self.set_x(self.w / 2 - 25)
        self.cell(0, 10, 'Confidencial - Uso Interno', 0, 0, 'C')
        self.set_x(self.w - 20)
        self.cell(0, 10, f'Página {self.page_no()}/{{nb}}', 0, 0, 'R')

    def titulo_seccion(self, titulo):
        self.set_font('Arial', 'B', 10)
        self.set_text_color(255, 255, 255)
        self.set_fill_color(0, 51, 102)
        self.cell(0, 8, f'  {titulo}', border=0, ln=1, align='L', fill=True)
        self.ln(4)

    def texto_normal(self, texto):
        self.set_text_color(0, 0, 0)
        line_height = 5

        # Split text by '**', keeping the delimiters
        parts = re.split(r'(\S*\*\*.*\*\*\S*)', texto) # Modified regex to ensure ** is not split if part of a word

        for part in parts:
            if part.startswith('**') and part.endswith('**'):
                # Text inside '**' should be bold
                self.set_font('Arial', 'B', 10)
                self.write(line_height, part[2:-2]) # write the content without '**'
            else:
                # Normal text
                self.set_font('Arial', '', 10)
                self.write(line_height, part)

        self.ln(line_height) # New line after the entire paragraph

def generar_pdf_ejecutivo(df_master, chapter_kpis_df, semana_corte):
    """Ensambla el documento PDF con la narrativa y los gráficos generados.
    """

    graficar_curva_s_dimensional(df_master, semana_corte)
    graficar_matriz_desempeno(chapter_kpis_df)

    pdf = InformePMO()
    pdf.alias_nb_pages()
    pdf.add_page()

    # Metadatos
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(40, 6, 'Proyecto:', 0, 0)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 6, 'Construcción Subestación Eléctrica 115kV/34.5kV', 0, 1)

    pdf.set_font('Arial', 'B', 12)
    pdf.cell(40, 6, 'Fecha de Corte:', 0, 0)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 6, f'Semana {semana_corte} ({datetime.now().strftime("%Y-%m-%d")})', 0, 1)
    pdf.ln(5)

    # Retrieve global KPI variables for narrative and totals table
    global_cpi = globals().get('global_cpi_for_pdf', 0.0)
    global_spi = globals().get('global_spi_for_pdf', 0.0)
    global_bac_total = globals().get('total_project_bac_mc', 0.0)

    # Sección 1
    pdf.titulo_seccion('1. RESUMEN EJECUTIVO Y MÉTRICAS GLOBALES')
    texto_resumen = (
        f"El proyecto presenta un Índice de Desempeño de Costo (CPI) de {global_cpi:.2f}, indicando un "
        "sobrecosto estructural originado en la fase de obras civiles. Simultáneamente, el Índice "
        f"de Desempeño de Cronograma (SPI) se sitúa en ~{global_spi:.2f}, reflejando un retraso en la ruta "
        "crítica debido a contingencias aduaneras en la importación de suministros mayores."
    )
    pdf.texto_normal(texto_resumen)

    # Tabla KPIs
    pdf.set_font('Arial', 'B', 11)
    pdf.cell(0, 8, 'Desempeño de KPIs por Capítulo', 0, 1, 'C')
    pdf.ln(2)

    col_widths = [40, 20, 20, 20, 20, 15, 15]
    headers = ['Capítulo', 'BAC', 'PV', 'EV', 'AC', 'CPI', 'SPI']
    total_table_width = sum(col_widths)
    start_x = (pdf.w - total_table_width) / 2

    pdf.set_x(start_x)
    pdf.set_fill_color(220, 220, 220)
    pdf.set_text_color(0, 0, 0)
    for i, header in enumerate(headers):
        pdf.cell(col_widths[i], 8, header, 1, 0, 'C', 1)
    pdf.ln()

    pdf.set_font('Arial', '', 9)
    for _, row in chapter_kpis_df.iterrows():
        pdf.set_x(start_x) # Retorno estricto al inicio de la Tabla 1
        pdf.cell(col_widths[0], 7, row['Capitulo'], 1, 0, 'L')
        pdf.cell(col_widths[1], 7, f"{row['BAC']:,d}", 1, 0, 'R')
        pdf.cell(col_widths[2], 7, f"{row['PV']:,d}", 1, 0, 'R')
        pdf.cell(col_widths[3], 7, f"{row['EV']:,d}", 1, 0, 'R')
        pdf.cell(col_widths[4], 7, f"{row['AC']:,d}", 1, 0, 'R')
        pdf.cell(col_widths[5], 7, f"{row['CPI']:.2f}", 1, 0, 'R')
        pdf.cell(col_widths[6], 7, f"{row['SPI']:.2f}", 1, 0, 'R')
        pdf.ln()

    # Totales Globales
    df_at_corte_final = df_master[df_master['Semana'] == semana_corte]
    total_pv = df_at_corte_final['PV'].sum()
    total_ev = df_at_corte_final['EV'].sum()
    total_ac = df_at_corte_final['AC'].sum()

    # These were previously recalculated locally, now using globally set values
    # global_cpi = total_ev / total_ac if total_ac != 0 else 0
    # global_spi = total_ev / total_pv if total_pv != 0 else 0

    # Correctly retrieve global_bac_total from the Monte Carlo results
    # global_bac_total = globals().get('total_project_bac_mc', 0.0) # Ensure it's a float
    # if global_bac_total == 0: # Fallback in case Monte Carlo was not run or didn't set it
    #     df_unique_wbs_bac = df_master[['WBS', 'BAC']].drop_duplicates(subset=['WBS'])
    #     global_bac_total = df_unique_wbs_bac['BAC'].sum()

    # Añadir depuración en generación de PDF
    TARGET_PV_GLOBAL = globals().get('TARGET_PV', 0)
    TARGET_EV_GLOBAL = globals().get('TARGET_EV', 0)
    TARGET_AC_GLOBAL = globals().get('TARGET_AC', 0)

    print(f"\n[DEBUG - Generación PDF] Total PV global para Curva S y tabla: {total_pv:,d}")
    print(f"[DEBUG - Generación PDF] Total EV global para Curva S y tabla: {total_ev:,d}")
    print(f"[DEBUG - Generación PDF] Total AC global para Curva S y tabla: {total_ac:,d}")
    print(f"[DEBUG - Generación PDF] TARGET PV (global): {TARGET_PV_GLOBAL:,d} | Diferencia: {total_pv - TARGET_PV_GLOBAL:,d}")
    print(f"[DEBUG - Generación PDF] TARGET EV (global): {TARGET_EV_GLOBAL:,d} | Diferencia: {total_ev - TARGET_EV_GLOBAL:,d}")
    print(f"[DEBUG - Generación PDF] TARGET AC (global): {TARGET_AC_GLOBAL:,d} | Diferencia: {total_ac - TARGET_AC_GLOBAL:,d}")


    pdf.set_x(start_x)
    pdf.set_fill_color(200, 200, 200)
    pdf.set_font('Arial', 'B', 9)
    pdf.cell(col_widths[0], 7, 'Totales', 1, 0, 'L', 1)
    pdf.cell(col_widths[1], 7, f"{global_bac_total:,.0f}", 1, 0, 'R', 1)
    pdf.cell(col_widths[2], 7, f"{total_pv:,d}", 1, 0, 'R', 1)
    pdf.cell(col_widths[3], 7, f"{total_ev:,d}", 1, 0, 'R', 1)
    pdf.cell(col_widths[4], 7, f"{total_ac:,d}", 1, 0, 'R', 1)
    pdf.cell(col_widths[5], 7, f"{global_cpi:.2f}", 1, 0, 'R', 1)
    pdf.cell(col_widths[6], 7, f"{global_spi:.2f}", 1, 0, 'R', 1)
    pdf.ln(10)

    # --- Sección 1.1: Proyecciones PMI (Estimaciones a la Conclusión) ---
    pdf.set_font('Arial', 'B', 11)
    pdf.cell(0, 8, '1.1 Proyecciones PMI (Estimaciones a la Conclusión)', 0, 1, 'L')
    pdf.ln(2)

    # Retrieve global KPI variables
    eac_cpi_global = globals().get('eac_cpi_global', 0.0)
    etc_cpi_global = globals().get('etc_cpi_global', 0.0)
    vac_cpi_global = globals().get('vac_cpi_global', 0.0)
    tcpi_bac_global = globals().get('tcpi_bac_global', np.nan)
    tcpi_eac_global = globals().get('tcpi_eac_global', np.nan)

    # Table for global KPIs
    kpi_headers = ['Métrica', 'Valor']
    kpi_col_widths = [80, 30]
    kpi_start_x = (pdf.w - sum(kpi_col_widths)) / 2

    pdf.set_x(kpi_start_x)
    pdf.set_fill_color(220, 220, 220)
    pdf.set_text_color(0, 0, 0)
    pdf.set_font('Arial', 'B', 9)
    for header in kpi_headers:
        pdf.cell(kpi_col_widths[kpi_headers.index(header)], 7, header, 1, 0, 'C', 1)
    pdf.ln()

    pdf.set_font('Arial', '', 9)
    kpi_data = [
        ['BAC (Budget At Completion)', f"${global_bac_total:,.0f}"],
        ['CPI Global (Cost Performance Index)', f"{global_cpi:.2f}"],
        ['SPI Global (Schedule Performance Index)', f"{global_spi:.2f}"],
        ['EAC (Estimación a la Conclusión)', f"${eac_cpi_global:,.0f}"],
        ['ETC (Estimación para Terminar)', f"${etc_cpi_global:,.0f}"],
        ['VAC (Variación a la Conclusión)', f"${vac_cpi_global:,.0f}"],
        ['TCPI (basado en BAC)', f"{tcpi_bac_global:.2f}" if not np.isnan(tcpi_bac_global) else "N/A"],
        ['TCPI (basado en EAC)', f"{tcpi_eac_global:.2f}" if not np.isnan(tcpi_eac_global) else "N/A"]
    ]

    for row in kpi_data:
        pdf.set_x(kpi_start_x)
        pdf.cell(kpi_col_widths[0], 7, row[0], 1, 0, 'L')
        pdf.cell(kpi_col_widths[1], 7, row[1], 1, 0, 'R')
        pdf.ln()
    pdf.ln(10)

    # --- Sección 2: ANÁLISIS TÉCNICO POR FRENTE CONSTRUCTIVO ---
    pdf.titulo_seccion('2. ANÁLISIS TÉCNICO POR FRENTE CONSTRUCTIVO')

    # Extract CPI and SPI values for each chapter with error handling
    try:
        ingenieria_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '1. Ingeniería'].iloc[0]
        ing_cpi = ingenieria_kpis['CPI']
        ing_spi = ingenieria_kpis['SPI']
    except IndexError:
        ing_cpi = 0.0
        ing_spi = 0.0

    try:
        obras_civiles_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '2. Obras Civiles'].iloc[0]
        oc_cpi = obras_civiles_kpis['CPI']
        oc_spi = obras_civiles_kpis['SPI']
    except IndexError:
        oc_cpi = 0.0
        oc_spi = 0.0

    try:
        suministros_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '3. Suministros'].iloc[0]
        sum_cpi = suministros_kpis['CPI']
        sum_spi = suministros_kpis['SPI']
    except IndexError:
        sum_cpi = 0.0
        sum_spi = 0.0

    try:
        montaje_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '4. Montaje'].iloc[0]
        mon_cpi = montaje_kpis['CPI']
        mon_spi = montaje_kpis['SPI']
    except IndexError:
        mon_cpi = 0.0
        mon_spi = 0.0

    try:
        pruebas_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '5. Pruebas'].iloc[0]
        prue_cpi = pruebas_kpis['CPI']
        prue_spi = pruebas_kpis['SPI']
    except IndexError:
        prue_cpi = 0.0
        prue_spi = 0.0

    # Insert dynamically generated narrative
    texto_ingenieria = (
        f"**INGENIERÍA** (CPI: {ing_cpi:.2f} | SPI: {ing_spi:.2f}): La fase de ingeniería ha progresado según lo planeado en cronograma, "
        "pero ha incurrido en costos adicionales, principalmente debido a estudios "
        "complementarios no previstos y ajustes en diseños de detalle."
    )
    pdf.texto_normal(texto_ingenieria)

    texto_civil = (
        f"**OBRAS CIVILES** (CPI: {oc_cpi:.2f} | SPI: {oc_spi:.2f}): Impacto financiero crítico. Se detectó un nivel "
        "freático excepcionalmente alto durante excavaciones de pórticos. Esto obligó al uso "
        "continuo de motobombas y entibados, erosionando las reservas de gestión de este paquete."
    )
    pdf.texto_normal(texto_civil)

    texto_suministros = (
        f"**SUMINISTROS DE POTENCIA** (CPI: {sum_cpi:.2f} | SPI: {sum_spi:.2f}): Retraso severo. Las órdenes de compra "
        "del Transformador de 30MVA y celdas GIS presentan demoras por huelgas portuarias e "
        "inspecciones aduaneras, desplazando los hitos FAT."
    )
    pdf.texto_normal(texto_suministros)

    texto_montaje = (
        f"**MONTAJE** (CPI: {mon_cpi:.2f} | SPI: {mon_spi:.2f}): La fase de montaje ha mantenido un buen ritmo, "
        "pero la disponibilidad de algunos equipos menores y la espera por la llegada de los "
        "suministros críticos podría impactar el cronograma futuro."
    )
    pdf.texto_normal(texto_montaje)

    texto_pruebas = (
        f"**PRUEBAS** (CPI: {prue_cpi:.2f} | SPI: {prue_spi:.2f}): Esta fase aún no ha iniciado actividades con costo significativo, "
        "manteniendo sus indicadores en cero. El inicio dependerá de la finalización de montaje y la disponibilidad de equipos."
    )
    pdf.texto_normal(texto_pruebas)

# Inserción de Gráfico de Cuadrantes (Salud del Proyecto) y Curva S lado a lado
    y_actual = pdf.get_y()

    # Ajuste: Se resta 2mm a la posición actual para subir los gráficos 3mm respecto al +1 anterior
    y_posicion_graficos = y_actual + 5

    # Calculate width for two plots with margins and spacing
    plot_width = 87.5  # (210 - (15*2) - 5) / 2 = 87.5mm

    # Se aplica y_posicion_graficos para elevar ambas imágenes
    pdf.image('temp_matriz_cpispi.png', x=15, y=y_posicion_graficos, w=plot_width)
    pdf.image('s_curve_plot.png', x=15 + plot_width + 5, y=y_posicion_graficos, w=plot_width)

    # Ajuste del salto de línea para compensar el movimiento hacia arriba de los gráficos
    pdf.ln(80)

    # --- Sección 3: ANÁLISIS DE RIESGO - SIMULACIÓN MONTECARLO ---
    pdf.titulo_seccion('3. ANÁLISIS DE RIESGO - SIMULACIÓN MONTECARLO')
    texto_advertencia_mc = (
        "Nota: La simulación Monte Carlo se realiza a nivel global del proyecto. "
        "Aunque proporciona una visión integral del riesgo de cronograma y costo, "
        "pueden existir desviaciones a nivel de capítulo o actividad que requieren "
        "análisis más detallados y específicos. Los resultados representan "
        "una estimación probabilística del desempeño general."
    )
    pdf.texto_normal(texto_advertencia_mc)

    # Retrieve global variables for probabilities and percentiles
    prob_success_path1 = globals().get('prob_success_path1', 0.0)
    prob_success_path2 = globals().get('prob_success_path2', 0.0)
    prob_exito = globals().get('prob_exito', 0.0)
    p10_tiempo = globals().get('p10_tiempo', 0.0)
    p50_tiempo = globals().get('p50_tiempo', 0.0)
    p90_tiempo = globals().get('p90_tiempo', 0.0)

    p10_costo = globals().get('p10_costo', 0.0)
    p50_costo = globals().get('p50_costo', 0.0)
    p90_costo = globals().get('p90_costo', 0.0)
    total_project_bac_mc = globals().get('total_project_bac_mc', 0.0)

    num_iteraciones = globals().get('num_iteraciones', 0)
    PLAZO = globals().get('PLAZO', 0)

# --- Inicio de la nueva sección: Texto descriptivo y Tablas Lado a Lado ---

    # 1. Título y Párrafo Narrativo de Parámetros
    pdf.set_font('Arial', 'B', 11)
    pdf.set_text_color(0, 51, 102)
    pdf.cell(0, 8, 'Resultados y Parametros de la Simulacion Monte Carlo'.encode('latin-1', 'replace').decode('latin-1'), 0, 1, 'L')
    pdf.ln()

    pdf.set_font('Arial', '', 10)
    pdf.set_text_color(0, 0, 0)

    # 2. Configuración de Coordenadas para Tablas Lado a Lado
    start_y_tables = pdf.get_y() - 4
    altura_fila = 6

    # =====================================================================
    # TABLA 1: RESULTADOS DE SIMULACIÓN (IZQUIERDA)
    # =====================================================================
    x_t1 = 20  # Margen izquierdo estándar
    # Anchos de columna ajustados para que los nombres de las métricas quepan bien (Total = 125mm)
    w_t1 = [30, 30, 30]
    headers_t1 = ['Escenarios', 'Tiempo en semanas', 'Costo']

    pdf.set_xy(x_t1, start_y_tables)
    pdf.set_fill_color(0, 51, 102)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font('Arial', 'B', 7)

    # Renderizado Encabezados Tabla 1
    for i in range(len(headers_t1)):
        pdf.cell(w_t1[i], altura_fila, headers_t1[i].encode('latin-1', 'replace').decode('latin-1'), 1, 0, 'C', fill=True)
    pdf.ln()

    # Matriz de Datos Tabla 1 (Con los nombres solicitados)
    datos_t1 = [
        ['Contractual', f"{PLAZO} sem", f"${total_project_bac_mc:,.0f}"],
        ['P10 (Optimista)', f"{p10_tiempo:.1f} sem", f"${p10_costo:,.0f}"],
        ['P50 (Más posible)', f"{p50_tiempo:.1f} sem", f"${p50_costo:,.0f}"],
        ['P90 (Conservador)', f"{p90_tiempo:.1f} sem", f"${p90_costo:,.0f}"]
    ]

    pdf.set_text_color(0, 0, 0)
    pdf.set_font('Arial', '', 7)
    for row in datos_t1:
        pdf.set_x(x_t1) # Retorno estricto al inicio de la Tabla 1
        pdf.cell(w_t1[0], altura_fila, str(row[0]).encode('latin-1', 'replace').decode('latin-1'), 1, 0, 'L')
        for i in range(1, 3):
            pdf.cell(w_t1[i], altura_fila, str(row[i]).encode('latin-1', 'replace').decode('latin-1'), 1, 0, 'C')
        pdf.ln()

    end_y_t1 = pdf.get_y()

    # =====================================================================
    # TABLA 2: RESUMEN DE PARÁMETROS (DERECHA)
    # =====================================================================
    x_t2 = 128  # Posición X (10 margen + 125 tabla 1 + 3 separación)
    w_t2 = [30, 30]  # Anchos de columna (Total = 62mm)
    headers_t2 = ['Parametro', 'Valor / Rango']

    y_current_t2 = start_y_tables

    pdf.set_xy(x_t2, y_current_t2)
    pdf.set_fill_color(0, 51, 102)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font('Arial', 'B', 7)

    # Renderizado Encabezados Tabla 2
    for i in range(len(headers_t2)):
        pdf.cell(w_t2[i], altura_fila, headers_t2[i].encode('latin-1', 'replace').decode('latin-1'), 1, 0, 'C', fill=True)

    y_current_t2 += altura_fila

    # Matriz de Datos Tabla 2
    datos_t2 = [
        ['Iteraciones', f"{num_iteraciones:,d}"],
        ['R. Obras Civiles', '28 - 32 - 48 sem'],
        ['R. Suministros', '35 - 40 - 55 sem']
    ]

    pdf.set_text_color(0, 0, 0)
    pdf.set_font('Arial', '', 7)
    for row in datos_t2:
        pdf.set_xy(x_t2, y_current_t2) # Control manual del eje Y para aislar de la Tabla 1
        pdf.cell(w_t2[0], altura_fila, str(row[0]).encode('latin-1', 'replace').decode('latin-1'), 1, 0, 'L')
        pdf.cell(w_t2[1], altura_fila, str(row[1]).encode('latin-1', 'replace').decode('latin-1'), 1, 0, 'C')
        y_current_t2 += altura_fila

    # 3. Restaurar cursor Y para elementos posteriores
    pdf.set_y(max(end_y_t1, y_current_t2) + 5)

    # 4. Título y Párrafo Narrativo de Parámetros
    pdf.set_font('Arial', 'B', 11)
    pdf.set_text_color(0, 51, 102)
    pdf.cell(0, 8, 'Graficos de Frecuencias y Rutas Criticas de la Simulacion Monte Carlo'.encode('latin-1', 'replace').decode('latin-1'), 0, 1, 'L')
    pdf.ln(5)

    pdf.set_font('Arial', '', 10)
    pdf.set_text_color(0, 0, 0)

# Inserción de Gráfico de Frecuencia Tiempo y Costo lado a lado
    y_actual = pdf.get_y()

    # Ajuste: Se resta 2mm a la posición actual para subir los gráficos 3mm respecto al +1 anterior
    y_posicion_graficos = y_actual - 2

    # Full page width (210mm page width - 15mm left margin - 15mm right margin = 180mm)
    full_page_width = 180

    # Se aplica y_posicion_graficos para elevar ambas imágenes
    pdf.image('montecarlo_tiempo_costo.png', x=15, y=y_posicion_graficos, w=full_page_width)

    # Ajuste del salto de línea para compensar el movimiento hacia arriba de los gráficos
    # Adjusted from 65 to 75 to fit the larger image, assuming proportional scaling
    pdf.ln(70)

    # --- Nueva Sección: PROBABILIDAD DE ÉXITO POR RUTAS CRÍTICAS ---

    pdf.add_page()
    y_actual_prob_chart = pdf.get_y()
    print(f"[DEBUG - PDF Generation] Y-coordinate for probabilidad_exito_rutas.png: {y_actual_prob_chart:.2f}")
    # DEBUG: Check if the file exists before attempting to embed it
    prob_img_path = 'probabilidad_exito_rutas.png'
    if os.path.exists(prob_img_path):
        print(f"[DEBUG - PDF Generation] Found {prob_img_path}. Attempting to embed.")
        pdf.image(prob_img_path, x=45, y=y_actual_prob_chart, w=120)
    else:
        print(f"[DEBUG - PDF Generation] WARNING: {prob_img_path} not found. Skipping embedding.")

    pdf.ln(75) # Adjust line break after this image

    # --- Sección 4: RECOMENDACIONES ESTRATÉGICAS ---

    pdf.titulo_seccion('4. RECOMENDACIONES ESTRATÉGICAS')

    pdf.set_font('Arial', 'B', 10)
    pdf.texto_normal("Basado en el análisis de desempeño y riesgos, se proponen las siguientes recomendaciones estratégicas para abordar los desafíos clave del proyecto:")
    pdf.ln()

    # Recomendaciones detalladas
    pdf.set_font('Arial', '', 10)

    # Ingeniería
    pdf.texto_normal("  *  **Ingeniería (CPI 0.54):** Realizar una auditoría detallada de cambios de alcance y variaciones en ingeniería. Implementar un control de cambios más estricto y procesos de revisión de diseños para contener sobrecostos.")
    pdf.ln(5)

    # Obras Civiles
    pdf.texto_normal("  *  **Obras Civiles (CPI 0.84, SPI 0.92):** Investigar a fondo la causa del alto nivel freático y la necesidad de motobombas/entibados. Desarrollar un plan de mitigación específico para futuros frentes con condiciones geotécnicas similares. Considerar una re-planificación de estas fases para reflejar la realidad del terreno.")
    pdf.ln(5)

    # Suministros
    pdf.texto_normal("  *  **Suministros (SPI 0.78):** Fortalecer la cadena de suministro con planes de contingencia para retrasos aduaneros y portuarios (huelgas, inspecciones). Evaluar proveedores alternativos o rutas logísticas más resilientes para el transformador y celdas GIS.")
    pdf.texto_normal("  *  **Suministros (SPI 0.78):** Fortalecer la cadena de suministro con planes de contingencia para retrasos aduaneros y portuarios (huelgas, inspecciones). Evaluar proveedores alternativos o rutas logísticas más resilientes para el transformador y celdas GIS.")
    pdf.ln(5)

    # Monte Carlo y Planificación
    pdf.texto_normal("  *  **Planificación y Riesgo:** Ajustar las estimaciones de plazo y costo del proyecto basándose en los percentiles P50 o P90 de la simulación Monte Carlo para una planificación más realista (~53 a 60 semanas y ~$5.4M a ~$6.1M). Focalizar la gestión de riesgos en la ruta crítica de Obras Civiles + Montaje + Pruebas (solo 43% de probabilidad de éxito a 52 semanas).")
    pdf.ln(5)

    # --- Sección 5: Nota Legal y Advertencia ---
    pdf.titulo_seccion('5. NOTA LEGAL Y ADVERTENCIA')
    disclaimer_text = (
        "Este informe ha sido generado mediante algoritmos automatizados. Los resultados representan "
        "estimaciones probabilísticas y no certezas absolutas. El usuario final asume la responsabilidad "
        "total por las acciones derivadas de este análisis."
    )
    pdf.texto_normal(disclaimer_text)
    pdf.ln(5)

    pdf.output(f'Informe_Direccion_Semana_{semana_corte}.pdf')

### 7. BLOQUE DE EJECUCIÓN PRINCIPAL

In [ ]:
# 0. CONFIGURACIÓN GLOBAL
SEM_CORTE = 20 # Semana de corte para el análisis del proyecto
PLAZO = 52 # Plazo total del proyecto en semanas para Monte Carlo

archivo_csv_datos = 'data_proyecto.csv'
db_file = 'project_data.db'
num_iteraciones = 10000 # Número de iteraciones para la simulación de Monte Carlo

# 2. Cargar y procesar el dataset
df_loaded = procesar_dataset_simulado(archivo_csv_datos)

# 3. Crear y poblar la base de datos SQLite
create_and_populate_sqlite_db(df_loaded, db_file)

# Cargar el modelo dimensional completo desde SQLite
df_modelo_dimensional = load_dimensional_model_from_sqlite(db_file)

# 4. Calcular indicadores por capítulo
chapter_kpis_df = calcular_indicadores_por_capitulo(df_modelo_dimensional, SEM_CORTE)

# 5. Calcular KPIs globales y guardarlos en el scope global
calcular_kpis_globales(df_modelo_dimensional, SEM_CORTE)

# 7. Generar reporte de Monte Carlo y sus gráficos
generar_reporte_montecarlo_completo(db_file, num_iteraciones)

# 8. Generar el informe PDF
pdf_filename = f'Informe_Direccion_Semana_{SEM_CORTE}.pdf'
generar_pdf_ejecutivo(df_modelo_dimensional, chapter_kpis_df, SEM_CORTE)
print(f"Informe PDF generado: '{pdf_filename}'")

# Descargar el informe PDF
files.download(pdf_filename)